### We load a real tabular dataset. <br>
### Goal: <br> - inspect shape, <br> - class balance, <br> - feature scale before touching a model. <br>

### I expect features on very different scales, which will matter for gradient descent.

In [1]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

data = load_breast_cancer()
X , y = data.data, data.target

print("X shape:", X.shape)
print("y shape:", y.shape)
print("class balance:", np.bincount(y))
print("\nfeature means (first 5):", X.mean(axis=0)[:5].round(2))
print("feature stds (first 5):", X.std(axis=0)[:5].round(2))

X shape: (569, 30)
y shape: (569,)
class balance: [212 357]

feature means (first 5): [1.4130e+01 1.9290e+01 9.1970e+01 6.5489e+02 1.0000e-01]
feature stds (first 5): [3.520e+00 4.300e+00 2.428e+01 3.516e+02 1.000e-02]


Splitting before scaling to prevent leakage. Stratifying to preserve the 212/357 class ratio in both sets. Fitting the scaler on train only, then transforming both.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.20,
    random_state = RANDOM_STATE,
    stratify = y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("train", X_train.shape, "test:", X_test.shape)
print("train class balance:", np.bincount(y_train))
print("test class balance:", np.bincount(y_test))
print("\nscaled train mean (first 5):", X_train.mean(axis=0)[:5].round(3))
print("scaled train std (first 5):", X_train.std(axis=0)[:5].round(3))
print("scaled test mean (first 5):", X_test.mean(axis=0)[:5].round(3))

train (455, 30) test: (114, 30)
train class balance: [170 285]
test class balance: [42 72]

scaled train mean (first 5): [-0.  0. -0.  0.  0.]
scaled train std (first 5): [1. 1. 1. 1. 1.]
scaled test mean (first 5): [0.086 0.048 0.085 0.092 0.072]


In [5]:
print("all 30 means — max abs:", np.abs(X_train.mean(axis=0)).max())
print("all 30 stds  — min:", X_train.std(axis=0).min(), "max:", X_train.std(axis=0).max())


all 30 means — max abs: 5.6770217342700865e-15
all 30 stds  — min: 0.9999999999999984 max: 1.000000000000001


In [6]:
print("min value in scaled train:", X_train.min().round(3))
print("max value in scaled train:", X_train.max().round(3))


min value in scaled train: -2.715
max value in scaled train: 11.658
